# 하이브리드 검색: RRF와 Convex Combination(CC)

Kiwi-BM25와 Chroma 의미 검색을 두 방식으로 결합합니다.

- **RRF**: 원점수 대신 순위를 결합하므로 점수 척도 차이에 강합니다.
- **CC**: 정규화된 sparse/dense 점수의 가중합으로, 점수의 크기 정보까지 씁니다.

외부 `langchain-teddynote` 구현 없이 알고리즘과 점수 보정을 직접 확인합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain-core==1.6.3" "langchain-openai==1.6.2" \
#   "langchain-chroma==1.1.0" "langchain-text-splitters==1.1.2" \
#   python-dotenv pymupdf rank-bm25 kiwipiepy numpy


In [ ]:
import getpass
import os
import unicodedata
from collections import defaultdict
from pathlib import Path
from uuid import uuid4

import numpy as np
import pymupdf
from dotenv import load_dotenv
from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


def bounded_cosine_relevance(distance: float) -> float:
    return max(0.0, min(1.0, 1.0 - distance / 2.0))


In [ ]:
def find_policy_pdf(data_dir: Path) -> Path:
    for path in data_dir.glob("*.pdf"):
        normalized = unicodedata.normalize("NFC", path.name)
        if "디지털정부혁신" in normalized:
            return path
    raise FileNotFoundError(
        "data 폴더에서 '디지털정부혁신' PDF를 찾지 못했습니다."
    )


def load_pdf(path: Path) -> list[Document]:
    with pymupdf.open(path) as pdf:
        return [
            Document(
                page_content=page.get_text("text"),
                metadata={"source": str(path), "page": page.number},
            )
            for page in pdf
            if page.get_text("text").strip()
        ]


pdf_path = find_policy_pdf(Path("data"))
pages = load_pdf(pdf_path)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    add_start_index=True,
)
chunks = splitter.split_documents(pages)
chunks = [
    Document(
        page_content=chunk.page_content,
        metadata={**chunk.metadata, "doc_id": f"chunk-{index}"},
    )
    for index, chunk in enumerate(chunks)
]
print(f"인덱싱할 청크 수: {len(chunks)}")


## Dense와 sparse 후보 생성


In [ ]:
dense_store = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name=f"cc-ensemble-{uuid4().hex}",
    ids=[chunk.metadata["doc_id"] for chunk in chunks],
    collection_configuration=CHROMA_CONFIGURATION,
    relevance_score_fn=bounded_cosine_relevance,
)

kiwi = Kiwi()


def tokenize(text: str) -> list[str]:
    return [token.form for token in kiwi.tokenize(text)]


bm25_index = BM25Okapi([tokenize(chunk.page_content) for chunk in chunks])


def dense_scored(query: str, k: int = 20) -> list[tuple[Document, float]]:
    return [
        (doc, float(score))
        for doc, score in dense_store.similarity_search_with_relevance_scores(
            query,
            k=min(k, len(chunks)),
        )
    ]


def sparse_scored(query: str, k: int = 20) -> list[tuple[Document, float]]:
    scores = bm25_index.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1][: min(k, len(chunks))]
    return [(chunks[index], float(scores[index])) for index in order]


## RRF와 CC 구현


In [ ]:
def minmax_scores(hits: list[tuple[Document, float]]) -> dict[str, float]:
    if not hits:
        return {}
    values = [score for _, score in hits]
    low, high = min(values), max(values)
    if np.isclose(low, high):
        return {doc.metadata["doc_id"]: 1.0 for doc, _ in hits}
    return {
        doc.metadata["doc_id"]: (score - low) / (high - low)
        for doc, score in hits
    }


def rrf_fusion(
    query: str,
    *,
    weights: tuple[float, float] = (0.5, 0.5),
    fetch_k: int = 20,
    result_k: int = 5,
    c: int = 60,
) -> list[Document]:
    rankings = [dense_scored(query, fetch_k), sparse_scored(query, fetch_k)]
    scores: defaultdict[str, float] = defaultdict(float)
    by_id: dict[str, Document] = {}
    for ranking, weight in zip(rankings, weights):
        for rank, (doc, _) in enumerate(ranking, start=1):
            doc_id = doc.metadata["doc_id"]
            by_id[doc_id] = doc
            scores[doc_id] += weight / (c + rank)
    ranked_ids = sorted(scores, key=scores.get, reverse=True)[:result_k]
    return [
        Document(
            page_content=by_id[doc_id].page_content,
            metadata={
                **by_id[doc_id].metadata,
                "fusion": "RRF",
                "fusion_score": scores[doc_id],
            },
        )
        for doc_id in ranked_ids
    ]


def cc_fusion(
    query: str,
    *,
    dense_weight: float = 0.5,
    fetch_k: int = 20,
    result_k: int = 5,
) -> list[Document]:
    if not 0 <= dense_weight <= 1:
        raise ValueError("dense_weight는 0과 1 사이여야 합니다.")
    dense_hits = dense_scored(query, fetch_k)
    sparse_hits = sparse_scored(query, fetch_k)
    dense_normalized = minmax_scores(dense_hits)
    sparse_normalized = minmax_scores(sparse_hits)

    by_id = {
        doc.metadata["doc_id"]: doc
        for doc, _ in [*dense_hits, *sparse_hits]
    }
    all_ids = set(dense_normalized) | set(sparse_normalized)
    scores = {
        doc_id: (
            dense_weight * dense_normalized.get(doc_id, 0.0)
            + (1.0 - dense_weight) * sparse_normalized.get(doc_id, 0.0)
        )
        for doc_id in all_ids
    }
    ranked_ids = sorted(scores, key=scores.get, reverse=True)[:result_k]
    return [
        Document(
            page_content=by_id[doc_id].page_content,
            metadata={
                **by_id[doc_id].metadata,
                "fusion": "CC",
                "fusion_score": scores[doc_id],
                "dense_normalized": dense_normalized.get(doc_id, 0.0),
                "sparse_normalized": sparse_normalized.get(doc_id, 0.0),
            },
        )
        for doc_id in ranked_ids
    ]


rrf_retriever = RunnableLambda(lambda query: rrf_fusion(query))
cc_retriever = RunnableLambda(
    lambda request: cc_fusion(
        request["query"],
        dense_weight=request.get("dense_weight", 0.5),
    )
)


## 결과 비교

CC는 반드시 점수 방향을 통일하고 정규화한 뒤 사용해야 합니다. 이 예제는 각
질의의 후보 집합 안에서 min-max 정규화하므로 점수는 질의 간 직접 비교할 수 없습니다.


In [ ]:
def print_hits(label: str, hits: list[Document]) -> None:
    print(f"\n[{label}]")
    for index, doc in enumerate(hits, start=1):
        print(
            f"{index}. score={doc.metadata['fusion_score']:.6f} "
            f"page={doc.metadata.get('page')}\n{doc.page_content[:300]}\n"
        )


query = "디지털 트랜스포메이션이란 무엇인가요?"
rrf_hits = rrf_retriever.invoke(query)
cc_hits = cc_retriever.invoke({"query": query, "dense_weight": 0.5})

print_hits("RRF", rrf_hits)
print_hits("CC (dense 0.5 + sparse 0.5)", cc_hits)
